# Bloc IA (WS1) : Exploratory Data Analysis (EDA)

## Context and Summary

Housing prices are an essential indicator of economic health and have significant implications for both homeowners and investors. In this Workshop, we will analyze a housing dataset to understand trends, distributions, and relationships between various factors affecting housing prices. We will walk through the data analysis process using Python libraries such as pandas, numpy, matplotlib, and seaborn, explaining each step in detail and posing questions to help interpret the results.

Nous allons essayer, dans le cadre de ce notebook, de prendre en main un jeu de données réel depuis l'import automatisé des fichers jusqu'à la préparation en vue d'alimenter un algorithme de _ML_.

Le _dataset_ présente des données immobilières californiennes. Il compte des variables telles que la population, le salaire médian, le prix moyen d'un logement, _etc_. Et ce pour chaque _block group_ (le _block group_ est la la plus petite division administrative aux Etats-Unis - entre 500 et 5000 personnes).

## Step-by-Step Analysis


### 1. Setting Up the Environment :

### Préparation de l'environnement

Ci-dessous quelques imports et précautions préalables à notre travail. Il n'est pas inutile de les parcourir.
Si nécessaire créer un bloc au démarrage pour installer toutes les librairies nécessaires en exécutant chacune leur tour les commandes suivantes:
- pip install numpy
- pip install pandas
- pip install Plotly, Matplotlib et SNS Seaborn

In [ ]:
!pip install numpy pandas plotly matplotlib seaborn missingno folium dash dash_core_components dash_html_components


First, we set up our environment by importing necessary libraries and configuring display settings for better readability of data and plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import tarfile
import os

# Configuring display settings
plt.rcParams['figure.figsize'] = (12, 9)
sns.set()
sns.set_context('talk')
np.set_printoptions(threshold=20, precision=2, suppress=True)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)
warnings.filterwarnings("ignore", category=FutureWarning)


### 2. Loading and Exploring the Housing Data

We start by loading the housing data from a .tgz archive. Then the extracted CSV file is loaded and and we will be exploring its structure.

In [ ]:
# Extract the .tgz file
file_path = 'housing.tgz'
extract_path = './housing_data'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

with tarfile.open(file_path) as tar:
    tar.extractall(path=extract_path)

# Assuming the extracted file is a CSV file named 'housing.csv'
csv_file_path = os.path.join(extract_path, 'housing.csv')

# Load the dataset
housing_data = pd.read_csv(csv_file_path)

The pd.read_csv function reads the CSV file into a pandas dataframe.

Question: What do you observe from the first few rows of the dataset?



**Answer:** Données numériques proches les unes des autres

### 3. Understanding the Data

Let's get a summary of the dataset to understand the distribution and identify any missing values.

In [ ]:
# Display basic information
housing_data.info()

In [ ]:
# Display summary statistics
housing_data.describe()

In [ ]:
# Display the first few rows of the dataset
housing_data.head()

The info method provides a concise summary of the dataframe, including the number of non-null entries and data types. The describe method gives statistical summaries for numerical columns.

3. La fonction [`value_count()`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.Series.value_counts.html?highlight=value_count) permet de connaître, par exemple, le nombre de valeurs différentes d'une [`Series`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.Series.html#pandas.Series) telle que `ocean_proximity`, qui semble être catégorielle:

In [ ]:
housing_data["ocean_proximity"].value_counts() #SOLUTION

Question: Are there any columns with missing values? How do you identify them?

**Answer:**

### 4. Handling Missing Values

For simplicity, let's fill missing values in numerical columns with the median value of the column. We will use MSNO library

In [ ]:
import missingno as msno
import seaborn as sns
import matplotlib.pyplot as plt

# Visualize missing data
msno.matrix(housing_data)
plt.show()

# Impute missing values
# For numerical columns, use median imputation
numerical_columns = housing_data.select_dtypes(include=['float64', 'int64']).columns
housing_data[numerical_columns] = housing_data[numerical_columns].fillna(housing_data[numerical_columns].median())

# For categorical columns, use mode imputation
categorical_columns = housing_data.select_dtypes(include=['object']).columns
housing_data[categorical_columns] = housing_data[categorical_columns].fillna(housing_data[categorical_columns].mode().iloc[0])

# Verify that there are no more missing values
housing_data.isnull().sum()


## 4. Univariate Analysis

Complete the following using the sns **histplot** method

In [ ]:
# Univariate analysis for numerical features
for column in numerical_columns:
        plt.figure(figsize=(10, 4))
        sns.histplot(housing_data[column], kde=True)
        plt.title(f'Distribution of {column}')
        plt.show()
    # à compléter
    # à compléter

# Univariate analysis for categorical features
for column in categorical_columns:
    plt.figure(figsize=(10, 4))
    sns.countplot(x=housing_data[column])
    plt.title(f'Distribution of {column}')
    plt.xticks(rotation=45)
    plt.show()


## Bivariate Analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Identify numerical and categorical columns
numerical_columns = housing_data.select_dtypes(include=['float64', 'int64']).columns
categorical_columns = housing_data.select_dtypes(include=['object']).columns
print('numerical_columns : ', numerical_columns)
print('categorical columns : ', categorical_columns)

Use the **pairplot** to discover linear trends

In [ ]:

# Scatter plots for pairs of numerical features
sns.pairplot(housing_data[numerical_columns])
plt.show()

In [ ]:
# Box plots for categorical vs numerical features
# Assuming 'ocean_proximity' is the categorical column
categorical_column = 'ocean_proximity'

for column in numerical_columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=housing_data[categorical_column], y=housing_data[column])
    plt.title(f'{column} by {categorical_column}')
    plt.show()


In [ ]:
# Correlation heatmap for numerical features
plt.figure(figsize=(12, 8))
sns.heatmap(housing_data[numerical_columns].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## Geo Info 

## Example 1: Hexbin Plot using Matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
plt.figure(figsize=(12, 8))
plt.hexbin(housing_data['longitude'], housing_data['latitude'], C=housing_data['median_house_value'], gridsize=50, cmap='YlOrRd', reduce_C_function=np.mean)
plt.colorbar(label='Mean House Price')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Hexbin Map of House Prices')
plt.show()


In [ ]:
import plotly.express as px

# Set Mapbox access token
px.set_mapbox_access_token("your_mapbox_token_here")

# Create a density heatmap
fig = px.density_mapbox(
    housing_data, 
    lat='latitude', 
    lon='longitude', 
    z='median_house_value', 
    radius=10,
    center=dict(lat=housing_data['latitude'].median(), lon=housing_data['longitude'].median()), 
    zoom=10,
    mapbox_style="stamen-terrain"
)

fig.update_layout(title='Density Heatmap of House Prices')

# Save the plot as an HTML file
fig.write_html('density_heatmap.html')

# Display the plot
fig.show()


## Example 3: Clustering & Heat Map using Folium

In [ ]:
import folium
from folium.plugins import MarkerCluster

# Create a map centered around the median coordinates
latitude = housing_data['latitude'].median()
longitude = housing_data['longitude'].median()
housing_map = folium.Map(location=[latitude, longitude], zoom_start=10)

# Add points to the map
marker_cluster = MarkerCluster().add_to(housing_map)

for idx, row in housing_data.iterrows():
    folium.Marker(location=[row['latitude'], row['longitude']],
                  popup=f"Price: {row['median_house_value']}").add_to(marker_cluster)

# Save the map to an HTML file
housing_map.save('housing_map.html')

# Display the map
housing_map


In [ ]:
from folium.plugins import HeatMap
# Folium map for geographic visualization
latitude = housing_data['latitude'].median()
longitude = housing_data['longitude'].median()
housing_map = folium.Map(location=[latitude, longitude], zoom_start=10)

# Prepare data for heatmap
heat_data = [[row['latitude'], row['longitude']] for index, row in housing_data.iterrows()]

# Add heatmap to the map
HeatMap(heat_data).add_to(housing_map)

# Save the map to an HTML file
housing_map.save('housing_heatmap.html')

# Display the map
housing_map

### Example 3 : Putting all together

In [ ]:
import folium
from folium.plugins import HeatMap, MarkerCluster
# Folium map for geographic visualization
latitude = housing_data['latitude'].median()
longitude = housing_data['longitude'].median()
housing_map = folium.Map(location=[latitude, longitude], zoom_start=10)

# Prepare data for heatmap with weights based on 'median_house_value'
heat_data = [[row['latitude'], row['longitude'], row['median_house_value']] for index, row in housing_data.iterrows()]

# Add heatmap to the map with weights
HeatMap(heat_data).add_to(housing_map)

# Add markers with popups for house prices
marker_cluster = MarkerCluster().add_to(housing_map)
for idx, row in housing_data.iterrows():
    folium.Marker(location=[row['latitude'], row['longitude']],
                  popup=f"Price: ${row['median_house_value']}",
                  tooltip=f"Price: ${row['median_house_value']}").add_to(marker_cluster)

# Save the map to an HTML file
housing_map.save('housing_heatmap_with_markers.html')

# Display the map
housing_map

# Advanced visualization with Dash

In [ ]:
import tarfile
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import dash
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
import folium
from folium.plugins import HeatMap, MarkerCluster
import missingno as msno

# Extract the .tgz file
file_path = 'housing.tgz'
extract_path = 'housing_data'

if not os.path.exists(extract_path):
    os.makedirs(extract_path)

with tarfile.open(file_path) as tar:
    tar.extractall(path=extract_path)

# Assuming the extracted file is a CSV file named 'housing.csv'
csv_file_path = os.path.join(extract_path, 'housing.csv')

# Load the dataset
housing_data = pd.read_csv(csv_file_path)

# Fill missing values
numerical_columns = housing_data.select_dtypes(include=['float64', 'int64']).columns
housing_data[numerical_columns] = housing_data[numerical_columns].fillna(housing_data[numerical_columns].median())
categorical_columns = housing_data.select_dtypes(include=['object']).columns
housing_data[categorical_columns] = housing_data[categorical_columns].fillna(housing_data[categorical_columns].mode().iloc[0])

# Create initial figures
# 1. Histogram of median house value
fig_hist = px.histogram(housing_data, x='median_house_value', nbins=50, title='Distribution of Median House Value')

# 2. Scatter plot of median house value vs median income
fig_scatter = px.scatter(housing_data, x='median_income', y='median_house_value', title='Median House Value vs Median Income')

# 3. Heatmap of correlation matrix
fig_corr = go.Figure(data=go.Heatmap(
    z=housing_data[numerical_columns].corr().values,
    x=numerical_columns,
    y=numerical_columns,
    colorscale='Viridis'
))

fig_corr.update_layout(title='Correlation Heatmap')

# 4. Density Heatmap using Plotly
px.set_mapbox_access_token("your_mapbox_token_here")
fig_density = px.density_mapbox(
    housing_data, 
    lat='latitude', 
    lon='longitude', 
    z='median_house_value', 
    radius=10,
    center=dict(lat=housing_data['latitude'].median(), lon=housing_data['longitude'].median()), 
    zoom=10,
    mapbox_style="stamen-terrain",
    title='Density Heatmap of House Prices'
)

# Create a folium map with heatmap
m = folium.Map(location=[housing_data['latitude'].median(), housing_data['longitude'].median()], zoom_start=10)
heat_data = [[row['latitude'], row['longitude'], row['median_house_value']] for index, row in housing_data.iterrows()]
HeatMap(heat_data).add_to(m)
m.save('housing_heatmap.html')

# Save the missing data matrix plot as an image
msno.matrix(housing_data).get_figure().savefig('missing_data_matrix.png')

# Initialize Dash app
app = dash.Dash(__name__)

# Define layout
app.layout = html.Div([
    html.H1('Housing Data Dashboard'),
    dcc.Tabs([
        dcc.Tab(label='Overview', children=[
            html.Div([
                html.H2('Univariate Analysis'),
                dcc.Graph(figure=fig_hist),
                html.H2('Bivariate Analysis'),
                dcc.Graph(figure=fig_scatter),
                html.H2('Correlation Heatmap'),
                dcc.Graph(figure=fig_corr),
            ])
        ]),
        dcc.Tab(label='Geographic Visualization', children=[
            html.Div([
                html.H2('Density Heatmap of House Prices'),
                dcc.Graph(figure=fig_density),
                html.H2('Interactive Folium Map'),
                html.Iframe(id='folium-map', srcDoc=open('housing_heatmap.html', 'r').read(), width='100%', height='600')
            ])
        ]),
        dcc.Tab(label='Missing Data Analysis', children=[
            html.Div([
                html.H2('Missing Data Matrix'),
                html.Img(src='missing_data_matrix.png', style={'width': '100%'})
            ])
        ])
    ])
])

# Run the app
if __name__ == '__main__':
    app.run(debug=True)
